# Thesis: Dual Sentiment Analysis (Expressed + Induced)
## XLM-RoBERTa Fine-tuning Experiments

**Πριν τρέξεις:**
1. Runtime → Change runtime type → **T4 GPU**
2. Ανέβασε τα αρχεία `filtered_dataset_clean.csv` και `viz_emotion_dataset_with_image_paths.pkl` στο Colab (αριστερά → Files → Upload)
3. Τρέξε όλα τα cells με **Runtime → Run All**

In [1]:
# ============================================================
# CELL 1 — Εγκατάσταση βιβλιοθηκών
# ============================================================
!pip install -q transformers datasets evaluate accelerate scikit-learn emoji sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — Imports + Mount Google Drive
# ============================================================
import re, os, json, time
import numpy as np
import pandas as pd
import torch
import emoji
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# Mount Google Drive για αποθήκευση αποτελεσμάτων
from google.colab import drive
drive.mount('/content/drive')

# Φάκελος αποθήκευσης
SAVE_DIR = '/content/drive/MyDrive/thesis_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(' Imports OK')
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
 Imports OK
GPU available: True
GPU: Tesla T4


In [3]:
# ============================================================
# CELL 3 — Load datasets + Merge + Normalize
# ============================================================

# Load
df_labels = pd.read_csv('filtered_dataset_clean.csv')
df_text   = pd.read_pickle('viz_emotion_dataset_with_image_paths.pkl')

# Filenames για merge
df_labels['fname'] = df_labels['image_path'].apply(lambda x: str(x).split('/')[-1].replace('.jpg',''))
df_text['fname']   = df_text['image_path'].apply(lambda x: str(x).split('/')[-1].replace('.jpg',''))

# Merge
df = df_labels.merge(df_text[['fname','text']], on='fname', how='left')
print(f'Dataset size: {df.shape}')

# Normalization
def normalize_text(text: str, keep_emojis: bool = True) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    if keep_emojis:
        text = emoji.demojize(text, delimiters=(' ', ' '))
        text = re.sub(r'_', ' ', text)
    else:
        text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_norm_emoji'] = df['text'].apply(lambda x: normalize_text(x, keep_emojis=True))
df['text_norm_clean'] = df['text'].apply(lambda x: normalize_text(x, keep_emojis=False))

# Labels
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label = {v: k for k, v in label2id.items()}

df_model = df[['text_norm_emoji','text_norm_clean',
               'voted_expressed','voted_induced']].dropna().copy()
df_model['label_expressed'] = df_model['voted_expressed'].map(label2id)
df_model['label_induced']   = df_model['voted_induced'].map(label2id)

# Split 70/15/15
train_df, temp_df = train_test_split(df_model, test_size=0.30,
                                      random_state=42,
                                      stratify=df_model['label_expressed'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                                      random_state=42,
                                      stratify=temp_df['label_expressed'])

# Class weights
def get_class_weights(labels_series):
    weights = compute_class_weight('balanced',
                                   classes=np.array([0,1,2]),
                                   y=labels_series.values)
    return torch.tensor(weights, dtype=torch.float)

weights_expressed = get_class_weights(train_df['label_expressed'])
weights_induced   = get_class_weights(train_df['label_induced'])

print(f' Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Weights expressed: {weights_expressed}')
print(f'Weights induced:   {weights_induced}')

Dataset size: (1881, 12)
 Train: 1316 | Val: 282 | Test: 283
Weights expressed: tensor([2.0989, 0.6285, 1.0725])
Weights induced:   tensor([1.8588, 3.5093, 0.4593])


In [4]:
# ============================================================
# CELL 4 — Dataset class + Helper functions
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
import evaluate

class SentimentDataset(Dataset):
    def __init__(self, df, text_col, label_col, tokenizer, max_len=128):
        self.texts     = df[text_col].tolist()
        self.labels    = df[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights.to(self.model.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Metrics
metric_acc = evaluate.load('accuracy')
metric_f1  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    f1  = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'f1_macro': f1}

print(' Classes & functions ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 Classes & functions ready


In [5]:
# Αποθήκευση ΜΟΝΟ metrics - όχι models
def run_experiment(model_name, text_col, task, class_weights):
    label_col = f'label_{task}'
    exp_name  = f"{model_name.split('/')[-1]}__{text_col}__{task}"
    print(f'\n{"="*60}')
    print(f' Experiment: {exp_name}')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds = SentimentDataset(train_df, text_col, label_col, tokenizer)
    val_ds   = SentimentDataset(val_df,   text_col, label_col, tokenizer)
    test_ds  = SentimentDataset(test_df,  text_col, label_col, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir='/tmp/r',
        num_train_epochs=4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='no',        # ΔΕΝ αποθηκεύει checkpoints
        load_best_model_at_end=False,
        fp16=True,
        report_to='none',
        logging_steps=50
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Evaluate on test set
    preds_out = trainer.predict(test_ds)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    y_true    = test_df[label_col].values

    report = classification_report(
        y_true, preds,
        target_names=['Negative','Neutral','Positive'],
        output_dict=True
    )

    result = {
        'experiment': exp_name,
        'model':      model_name,
        'text':       text_col,
        'task':       task,
        'accuracy':   round(report['accuracy'], 4),
        'f1_macro':   round(report['macro avg']['f1-score'], 4),
        'f1_negative':round(report['Negative']['f1-score'], 4),
        'f1_neutral': round(report['Neutral']['f1-score'], 4),
        'f1_positive':round(report['Positive']['f1-score'], 4),
    }

    all_results.append(result)
    print(f' Acc={result["accuracy"]} | F1_macro={result["f1_macro"]}')
    print(f'   Neg={result["f1_negative"]} | Neu={result["f1_neutral"]} | Pos={result["f1_positive"]}')

    return result

print(' Νέα run_experiment() έτοιμη - χωρίς αποθήκευση models!')

 Νέα run_experiment() έτοιμη - χωρίς αποθήκευση models!


In [7]:
all_results = []

In [8]:
# ============================================================
# CELL 6 — Τρέξε όλα τα experiments
# ============================================================
# Κάθε experiment παίρνει ~3-5 λεπτά με GPU T4
# Συνολικά: ~30-40 λεπτά για όλα

MODELS = [
    'xlm-roberta-base',
    'cardiffnlp/twitter-xlm-roberta-base-sentiment',
    'nlpaueb/bert-base-greek-uncased-v1',
]

TEXT_COLS = ['text_norm_emoji', 'text_norm_clean']
TASKS     = ['expressed', 'induced']

for model_name in MODELS:
    for text_col in TEXT_COLS:
        for task in TASKS:
            weights = weights_expressed if task == 'expressed' else weights_induced
            run_experiment(model_name, text_col, task, weights)

print('\n Όλα τα experiments τελείωσαν!')


 Experiment: xlm-roberta-base__text_norm_emoji__expressed


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.968805,0.404255,0.377662
2,1.069325,0.917749,0.468085,0.470965
3,0.951713,0.892915,0.489362,0.488451
4,0.851369,0.887615,0.457447,0.452562


 Acc=0.4311 | F1_macro=0.4307
   Neg=0.4603 | Neu=0.34 | Pos=0.4917

 Experiment: xlm-roberta-base__text_norm_emoji__induced


...

 Acc=0.4276 | F1_macro=0.4276
   Neg=0.4262 | Neu=0.4141 | Pos=0.4424

 Experiment: bert-base-greek-uncased-v1__text_norm_clean__induced


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.061703,0.251773,0.203928
2,1.115903,1.000829,0.645390,0.399884
3,1.008614,0.968264,0.652482,0.479579
4,0.913111,0.968772,0.673759,0.483408


 Acc=0.6608 | F1_macro=0.4883
   Neg=0.5 | Neu=0.1724 | Pos=0.7926

 Όλα τα experiments τελείωσαν!


In [9]:
# ============================================================
# CELL 7 — Αποθήκευση & Εμφάνιση αποτελεσμάτων
# ============================================================

results_df = pd.DataFrame(all_results)

# Αποθήκευση στο Drive
results_path = f'{SAVE_DIR}/bert_results.csv'
results_df.to_csv(results_path, index=False)
print(f' Results saved to: {results_path}')

# Εμφάνιση
print('\n=== ΣΥΝΟΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ ===')
display(results_df.sort_values('f1_macro', ascending=False))

 Results saved to: /content/drive/MyDrive/thesis_models/bert_results.csv

=== ΣΥΝΟΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ ===


,experiment,model,text,task,accuracy,f1_macro,f1_negative,f1_neutral,f1_positive
3,xlm-roberta-base__text_norm_clean__induced,xlm-roberta-base,text_norm_clean,induced,0.7385,0.5664,0.5155,0.3188,0.8650
1,xlm-roberta-base__text_norm_emoji__induced,xlm-roberta-base,text_norm_emoji,induced,0.7102,0.5383,0.5179,0.2623,0.8346
7,twitter-xlm-roberta-base-sentiment__text_norm_...,cardiffnlp/twitter-xlm-roberta-base-sentiment,text_norm_clean,induced,0.6890,0.5306,0.5812,0.1892,0.8213
5,twitter-xlm-roberta-base-sentiment__text_norm_...,cardiffnlp/twitter-xlm-roberta-base-sentiment,text_norm_emoji,induced,0.6678,0.5198,0.5455,0.2162,0.7978
6,twitter-xlm-roberta-base-sentiment__text_norm_...,cardiffnlp/twitter-xlm-roberta-base-sentiment,text_norm_clean,expressed,0.5265,0.5124,0.4643,0.5703,0.5026
4,twitter-xlm-roberta-base-sentiment__text_norm_...,cardiffnlp/twitter-xlm-roberta-base-sentiment,text_norm_emoji,expressed,0.5194,0.5111,0.4746,0.5462,0.5126
11,bert-base-greek-uncased-v1__text_norm_clean__i...,nlpaueb/bert-base-greek-uncased-v1,text_norm_clean,induced,0.6608,0.4883,0.5000,0.1724,0.7926
9,bert-base-greek-uncased-v1__text_norm_emoji__i...,nlpaueb/bert-base-greek-uncased-v1,text_norm_emoji,induced,0.6749,0.4842,0.4923,0.1509,0.8094
2,xlm-roberta-base__text_norm_clean__expressed,xlm-roberta-base,text_norm_clean,expressed,0.4629,0.4634,0.4655,0.4561,0.4685
0,xlm-roberta-base__text_norm_emoji__expressed,xlm-roberta-base,text_norm_emoji,expressed,0.4311,0.4307,0.4603,0.3400,0.4917


Expressed sentiment: Το twitter-xlm-roberta + clean κερδίζει με F1=0.512 — λογικό γιατί είναι ήδη trained σε Twitter data.
Induced sentiment: Το xlm-roberta-base + clean κερδίζει με F1=0.566 — ενδιαφέρον, το απλό base model τα πάει καλύτερα!
Μεγάλο πρόβλημα: Η Neutral class έχει πολύ χαμηλό F1 παντού (0.15-0.57) — αυτό είναι αναμενόμενο λόγω της imbalance αλλά είναι κάτι που πρέπει να αναφέρεις στη διπλωματική.

Τα βασικά συμπεράσματα από τα BERT experiments:
Expressed: twitter-xlm-roberta + clean → F1=0.512 (καλύτερο)
Induced: xlm-roberta-base + clean → F1=0.566 (καλύτερο)
Γενικά τα νούμερα είναι μέτρια (0.41-0.57) — αυτό είναι αναμενόμενο γιατί το expressed/induced sentiment είναι πολύ πιο δύσκολο από κλασικό sentiment analysis. Και αυτό ακριβώς δικαιολογεί γιατί χρειαζόμαστε LLMs με custom definitions!